In [60]:
import pandas as pd 
import os 
from skimage.transform import resize
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier  

In [61]:
from pathlib import Path
import os

print("cwd:", Path.cwd())
debug_dir = Path("/Users/imac/Computer_vision/IMAGES")
print("debug_dir:", debug_dir)
print("exists:", debug_dir.exists())
print("cwd entries:", [p.name for p in Path.cwd().iterdir()][:10])

cwd: /content
debug_dir: /Users/imac/Computer_vision/IMAGES
exists: False
cwd entries: ['.config', 'drive', 'sample_data']


In [62]:
# Option B: Mount Google Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [63]:
# Search for cats/dogs folders in Drive or /content
from pathlib import Path

search_roots = [Path('/content/drive/MyDrive'), Path('/content')]
drive_images_dir = None

for root in search_roots:
    if not root.exists():
        continue
    for cats_dir in root.rglob('cats'):
        parent = cats_dir.parent
        if (parent / 'dogs').exists():
            drive_images_dir = parent
            break
    if drive_images_dir:
        break

print('drive_images_dir:', drive_images_dir)

drive_images_dir: /content/drive/MyDrive/IMAGES


In [66]:
from pathlib import Path
import os

Categories=['cats','dogs']
flat_data_arr=[]
target_arr=[]

# Resolve dataset directory across local/colab/workspace setups
candidate_dirs = [
    Path('/Users/imac/Computer_vision/IMAGES'),
    Path('/content/Computer_vision/IMAGES'),
    Path('/content/IMAGES'),
    Path('/content/drive/MyDrive/IMAGES'),
    Path('/content/drive/MyDrive/Computer_vision/IMAGES'),
    Path.cwd() / 'IMAGES',
    Path.cwd().parent / 'IMAGES',
    Path.cwd().parent / 'Computer_vision' / 'IMAGES',
]
if 'drive_images_dir' in globals() and drive_images_dir:
    candidate_dirs.insert(0, drive_images_dir)

datadir = next((p for p in candidate_dirs if p.exists()), None)
if datadir is None:
    raise FileNotFoundError(
        "Could not find IMAGES folder. Checked: " + ", ".join(str(p) for p in candidate_dirs)
    )

for i in Categories:
    print(f'loading.............category:{i}')
    path = datadir / i
    if not path.exists():
        raise FileNotFoundError(f"Missing folder: {path}")
    for img in os.listdir(path):
        img_array=imread(os.path.join(path,img))
        img_resized=resize(img_array,(150,150,3))
        flat_data_arr.append(img_resized.flatten())
        target_arr.append(Categories.index(i))
flat_data_arr=np.array(flat_data_arr)
target_arr=np.array(target_arr)

loading.............category:cats
loading.............category:dogs


In [67]:
df=pd.DataFrame(flat_data_arr)
df['Target']=target_arr
df.head()

,0,1,2,3,4,5,6,7,8,9,...,67491,67492,67493,67494,67495,67496,67497,67498,67499,Target
0,0.442843,0.329364,0.258653,0.403134,0.296888,0.222560,0.432189,0.326371,0.251990,0.473403,...,0.293641,0.297562,0.274115,0.310316,0.314056,0.294900,0.296532,0.296655,0.288443,0
1,0.889991,0.889991,0.882148,0.872327,0.872327,0.864484,0.937672,0.937672,0.929829,0.950881,...,0.493368,0.469851,0.485535,0.484552,0.465447,0.478494,0.494909,0.490377,0.492509,0
2,0.999872,0.996677,0.999045,0.999872,0.996677,0.999045,0.999872,0.996677,0.999045,0.999872,...,0.431881,0.440174,0.398258,0.431078,0.437905,0.401746,0.443401,0.448355,0.423868,0
3,0.264409,0.205586,0.115389,0.270531,0.211708,0.121512,0.271997,0.213173,0.122977,0.270592,...,0.615717,0.580309,0.552346,0.616633,0.585053,0.564827,0.557323,0.518419,0.513974,0
4,0.852261,0.833049,0.881553,0.820220,0.801414,0.815409,0.691532,0.694811,0.720729,0.319682,...,0.356865,0.160787,0.031375,0.356037,0.159958,0.030547,0.353046,0.156967,0.027556,0


In [68]:
df.shape

(1399, 67501)

In [69]:
x=df.iloc[:,:-1]
y=df.iloc[:,-1]

In [70]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=77,stratify=y)

In [71]:
param_gram={'C':[0.1,1,10],'kernel':['rbf','linear'],'gamma':['scale','auto']}
svc=svm.SVC(probability=True)
model=GridSearchCV(svc,param_gram,cv=5)

In [ ]:
model.fit(x_train,y_train)

In [ ]:
y_pred=model.predict(x_test)
y_pred.shape

In [ ]:
accuracy_score(y_test,y_pred)
print(classification_report(y_test,y_pred))
print(f"The accuracy score is {accuracy_score(y_test,y_pred)*100:.2f}%")

In [ ]:
import seaborn as sns
print(classification_report(y_test,y_pred)) 
sns.heatmap(confusion_matrix(y_test,y_pred),annot=True) 
plt.show()  
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")    

In [ ]:
print(classification_report(y_test,y_pred)) 
sns.heatmap(confusion_matrix(y_test,y_pred),annot=True) 